# Descenso de gradiente: el impacto de escalar las variables

Van a implementar y correr un descenso de gradiente de principio a fin para un modelo de regresión lineal con **dos variables en escalas muy distintas** — el mismo ejemplo de tu diapositiva "Descenso de gradiente – Impacto de escalas": tamaño de casa (m²) y número de habitaciones, para predecir el precio.

El modelo es: $\hat{y} = w_1 x_1 + w_2 x_2$, donde $x_1$ = tamaño y $x_2$ = número de habitaciones. Van a usar un contour plot (el costo $J(w_1, w_2)$ visto desde arriba, como un mapa de curvas de nivel) para ver, literalmente, el camino que sigue el descenso de gradiente hasta llegar al mínimo.

**Truco técnico:** en vez de cargar con un tercer parámetro $b$ (bias), vamos a centrar $x_1$, $x_2$ y $y$ (restarles su media) antes de entrenar. Esto es matemáticamente equivalente a tener un modelo con bias — pero nos deja con solo dos parámetros, $w_1$ y $w_2$, así que el contour plot se puede graficar directo en 2D.

In [ ]:
# ============================================================
# PREPARACIÓN — datos sintéticos
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.RandomState(7)
n = 25

tamano = rng.uniform(50, 300, n)              # tamaño de la casa, en m^2
habitaciones = rng.randint(1, 6, n).astype(float)  # número de habitaciones (1 a 5)

w1_real, w2_real = 18.0, 150.0
ruido = rng.normal(0, 200, n)
precio = w1_real * tamano + w2_real * habitaciones + 3000 + ruido  # precio, miles de MXN

print(f"tamano:       media={tamano.mean():7.1f}   std={tamano.std():7.1f}")
print(f"habitaciones: media={habitaciones.mean():7.1f}   std={habitaciones.std():7.1f}")
print(f"\nRazón de escalas (std tamaño / std habitaciones): {tamano.std()/habitaciones.std():.1f}x")

Ahí está el problema antes de entrenar nada: `tamano` varía en decenas/cientos, `habitaciones` varía entre 1 y 5 — una razón de escalas de más de 50x. Vamos a ver qué le hace eso al descenso de gradiente.

In [ ]:
# ============================================================
# Funciones auxiliares
# ============================================================
def calcular_costo(x1, x2, y, w1, w2):
    pred = w1 * x1 + w2 * x2
    return np.mean((pred - y) ** 2)

def calcular_gradiente(x1, x2, y, w1, w2):
    pred = w1 * x1 + w2 * x2
    error = pred - y
    dw1 = 2 * np.mean(error * x1)
    dw2 = 2 * np.mean(error * x2)
    return dw1, dw2

def descenso_gradiente(x1, x2, y, w1_inicial, w2_inicial, learning_rate, iteraciones):
    w1, w2 = w1_inicial, w2_inicial
    historial_costo = []
    historial_w = [(w1, w2)]
    for i in range(iteraciones):
        dw1, dw2 = calcular_gradiente(x1, x2, y, w1, w2)
        w1 -= learning_rate * dw1
        w2 -= learning_rate * dw2
        costo = calcular_costo(x1, x2, y, w1, w2)
        historial_costo.append(costo)
        historial_w.append((w1, w2))
        if not np.isfinite(costo) or costo > 1e12:
            print(f"  Se disparó en la iteración {i} (costo={costo:.2e}) — diverge, learning_rate demasiado grande.")
            break
    return w1, w2, historial_costo, historial_w

def graficar_contorno(x1, x2, y, historial_w, ax, w1_range, w2_range, titulo, paso=1):
    w1_vals = np.linspace(*w1_range, 150)
    w2_vals = np.linspace(*w2_range, 150)
    W1, W2 = np.meshgrid(w1_vals, w2_vals)
    Z = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            Z[i, j] = calcular_costo(x1, x2, y, W1[i, j], W2[i, j])
    ax.contour(W1, W2, Z, levels=15)
    hx = [p[0] for p in historial_w[::paso]]
    hy = [p[1] for p in historial_w[::paso]]
    ax.plot(hx, hy, '-o', color='red', markersize=2, linewidth=1)
    ax.set_xlabel("w1 (tamaño)")
    ax.set_ylabel("w2 (habitaciones)")
    ax.set_title(titulo)

print("Listo.")

## Corrida 1 — sin escalar

Vamos a centrar los datos (restar la media) pero SIN estandarizar — `tamano` y `habitaciones` se quedan en sus unidades originales, con esa razón de escalas de 50x.

In [ ]:
tamano_c = tamano - tamano.mean()
habitaciones_c = habitaciones - habitaciones.mean()
precio_c = precio - precio.mean()

# Este learning rate está deliberadamente cerca del máximo que w1 tolera sin
# diverger.
# Se las dejamos así, cerca del límite, a propósito -- es justo donde se ve
# mejor el zigzagueo.
w1_final, w2_final, costo_hist, w_hist = descenso_gradiente(
    tamano_c, habitaciones_c, precio_c, w1_inicial=0.0, w2_inicial=0.0,
    learning_rate=1.6e-4, iteraciones=2000
)

print(f"Después de 2000 iteraciones: w1={w1_final:.2f}  w2={w2_final:.2f}")
print(f"(valores reales usados para generar los datos: w1={w1_real}  w2={w2_real})")
print(f"Costo final: {costo_hist[-1]:.1f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(costo_hist)
axes[0].set_xlabel("Iteración"); axes[0].set_ylabel("Costo"); axes[0].set_title("Costo vs. iteración (sin escalar)")
graficar_contorno(tamano_c, habitaciones_c, precio_c, w_hist, axes[1],
                   w1_range=(-10, 40), w2_range=(-500, 800), titulo="Contorno -- vista completa", paso=20)
# Zoom: mismo contorno pero acercado a donde realmente pasa la acción

graficar_contorno(tamano_c, habitaciones_c, precio_c, w_hist[:60], axes[2],
                   w1_range=(10, 26), w2_range=(-5, 30), titulo="Contorno -- zoom, primeros 60 pasos", paso=1)
plt.tight_layout()
plt.show()

# Ahora veamos w1 y w2 por separado contra la iteración, solo los primeros 60
# pasos -- aquí es donde se aprecia mejor lo que le pasa a cada uno.
# w1 se ve dos veces: completo, y con zoom en el eje y -- el primer salto es
# tan grande que si no hacemos zoom, el zigzagueo de los pasos siguientes casi
# no se alcanza a ver (se aplasta contra el eje x).
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
primeros = 60
w1_traza = [p[0] for p in w_hist[:primeros]]
w2_traza = [p[1] for p in w_hist[:primeros]]

axes[0].plot(w1_traza, marker='o', markersize=3)
axes[0].axhline(w1_real, color='green', linestyle='--', label='valor real')
axes[0].set_title("w1 (tamaño) -- vista completa")
axes[0].set_xlabel("Iteración"); axes[0].legend()

axes[1].plot(w1_traza, marker='o', markersize=3)
axes[1].axhline(w1_real, color='green', linestyle='--', label='valor real')
axes[1].set_ylim(10, 26)
axes[1].set_title("w1 (tamaño) -- con ZOOM (eje y: 10 a 26)")
axes[1].set_xlabel("Iteración"); axes[1].legend()

axes[2].plot(w2_traza, marker='o', markersize=3, color='orange')
axes[2].axhline(w2_real, color='green', linestyle='--', label='valor real')
axes[2].set_title("w2 (habitaciones) -- vista completa")
axes[2].set_xlabel("Iteración"); axes[2].legend()
plt.tight_layout()
plt.show()

**Las gráficas con "zoom" son las que realmente muestran el zigzagueo.** En la vista completa del contorno (primera fila, panel de en medio), el camino rojo parece un salto limpio hacia arriba, sin nada raro — pero es un espejismo de la escala: con un rango de `w2` tan grande (-500 a 800), los rebotes de `w1` quedan comprimidos en una rayita vertical casi invisible. Al acercar la cámara (panel de la derecha, mismo contorno pero acotado y sin saltarse ningún paso) se ve clarísimo un patrón en forma de abanico: el camino rebota de un lado a otro cruzando el valle angosto varias veces antes de asentarse en el centro.

Lo mismo pasa con `w1` contra la iteración (segunda fila): sin el zoom, el primer salto (de 0 a 34) aplasta todo lo que pasa después contra el eje — con el eje acotado entre 10 y 26 se ve que `w1` no converge suave, rebota varias veces arriba y abajo de su valor real antes de asentarse (eso es el zigzagueo: en la dirección empinada, cualquier paso lo suficientemente grande para avanzar rápido se pasa de largo del mínimo, corrige, se vuelve a pasar del otro lado, y así, con la oscilación cada vez más chica). Mientras tanto, `w2` (a la derecha) casi ni se mueve en esos mismos 60 pasos — su dirección es tan plana que necesita miles de iteraciones para recorrer la misma distancia relativa que `w1` recorre en unos cuantos pasos.

Ese es el problema real de tener variables en escalas tan distintas: un solo learning rate compartido no le puede quedar bien a las dos direcciones al mismo tiempo — el que hace que `w1` casi diverja (rebotando) hace que `w2` avance a paso de tortuga.

**¿Y si subimos el learning rate un poco más, para que `w2` avance más rápido?** Pruébenlo — la siguiente celda usa un learning rate apenas un 40% más grande.

In [ ]:
# Learning rate apenas un poco más grande...
w1_x, w2_x, costo_hist_x, _ = descenso_gradiente(
    tamano_c, habitaciones_c, precio_c, w1_inicial=0.0, w2_inicial=0.0,
    learning_rate=2.2e-4, iteraciones=12
)
print("Primeros costos:", [f"{c:.2e}" for c in costo_hist_x[:8]])

El costo explota en cuestión de pocas iteraciones — apenas un 40% más de learning rate fue suficiente para cruzar la línea entre "zigzaguea pero converge" y "diverge sin remedio". El margen es angostísimo, y esa es justo la consecuencia de la escala tan dispareja: la dirección de `w1` es tan sensible que casi no hay espacio entre "demasiado lento" y "explota".

## Tu turno — arregla el problema con escalado

Estandariza `tamano_c` y `habitaciones_c` (réstales su propia media — ya está hecho arriba — y divide entre su desviación estándar). Puedes hacerlo a mano o con `StandardScaler` de sklearn.

In [ ]:
# TODO: Crea tamano_z y habitaciones_z, la versión estandarizada
# (media 0, desviación estándar 1) de tamano_c y habitaciones_c.
#
# tamano_z = ...
# habitaciones_z = ...


## Corrida 2 — con escalado

Ahora corran el mismo `descenso_gradiente`, pero sobre `tamano_z` y `habitaciones_z`. Prueben un learning rate bastante más grande (por ejemplo `0.05`) y muchas menos iteraciones (`300` en vez de `2000`).

In [ ]:
# Corran aquí descenso_gradiente sobre tamano_z, habitaciones_z, con learning_rate=0.05 e iteraciones=300
# Después grafiquen el costo y el contorno, igual que en la Corrida 1
# (para el contorno, prueben con w1_range=(-200, 1800), w2_range=(-400, 900))



## Para discutir

1. Compara los dos contour plots. ¿Por qué el de la Corrida 1 sale tan alargado, y el de la Corrida 2 (con escalado) sale mucho más redondo?

2. En la Corrida 1 necesitaron 2000 iteraciones y ni así `w2` llegó a su valor real. En la Corrida 2, ¿cuántas iteraciones necesitaron para llegar a un resultado igual de bueno (o mejor)?

3. ¿Qué learning rate máximo aproximado pudieron usar en la Corrida 2 antes de que divergiera? Compárenlo con el `3e-4` que ya vieron que hacía explotar la Corrida 1.

4. En tus propias palabras: ¿por qué estandarizar las variables antes de entrenar con descenso de gradiente no es solo una "buena práctica" opcional, sino algo que puede ser la diferencia entre que el modelo converja en segundos o casi nunca?

5. Ya has visto `StandardScaler` dentro de un `Pipeline` en varias actividades anteriores (Ridge/Lasso, aprendizaje con SGDRegressor). ¿Ahora entiendes mejor por qué siempre aparece ahí?